# Directional Trading Bot: Polymarket Signal → Kalshi Execution

This bot polls Polymarket for price signals (higher volume = better price discovery) and executes trades on Kalshi when the prices diverge enough to cover fees.

## Strategy:
1. **Poll Polymarket** for the "true" price (higher liquidity, better price discovery)
2. **Compare to Kalshi** ask/bid prices
3. **Execute on Kalshi** if Poly price suggests Kalshi is mispriced after fees
4. **Account for fees**: Taker 7%, Maker 1.75%

In [2]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import time
import math
import threading
import uuid
import logging
import httpx
from collections import deque
from datetime import datetime
from dataclasses import dataclass
from typing import Optional, Dict, Tuple

from kalshi_utils.client_wrapper import KalshiWrapped

# Setup logging
def setup_logger(notebook_dir: str) -> logging.Logger:
    log_dir = os.path.join(notebook_dir, "logs")
    os.makedirs(log_dir, exist_ok=True)
    
    session_id = str(uuid.uuid4())[:8]
    date_str = datetime.now().strftime("%Y-%m-%d")
    log_filename = f"directional_{date_str}_{session_id}.log"
    log_path = os.path.join(log_dir, log_filename)
    
    logger = logging.getLogger(f"directional_{session_id}")
    logger.setLevel(logging.DEBUG)
    logger.handlers = []
    
    fh = logging.FileHandler(log_path)
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
    logger.addHandler(fh)
    
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(logging.Formatter('%(message)s'))
    logger.addHandler(ch)
    
    logger.info(f"=== Logging started: {log_path} ===")
    return logger

NOTEBOOK_DIR = os.getcwd()
print("Setup complete")

Setup complete


In [3]:
# Initialize Kalshi client
kalshi_wrapped = KalshiWrapped()

Client Created


In [4]:
# ============================================================================
# KALSHI FEE CALCULATIONS
# ============================================================================
def _round_up_cent(x: float) -> float:
    return math.ceil(x * 100.0) / 100.0

def kalshi_fee_total(contracts: int, price: float, maker: bool = False) -> float:
    """
    Kalshi fee: round_up(rate * C * P * (1-P))
    Taker: 7%, Maker: 1.75%
    """
    rate = 0.0175 if maker else 0.07
    return _round_up_cent(rate * contracts * price * (1.0 - price))

def kalshi_fee_per_contract(contracts: int, price: float, maker: bool = False) -> float:
    return kalshi_fee_total(contracts, price, maker=maker) / contracts

def kalshi_all_in_buy_cost(ask_price: float, contracts: int, maker: bool = False) -> float:
    """Cost per contract to buy at ask, including fees"""
    return ask_price + kalshi_fee_per_contract(contracts, ask_price, maker=maker)

def kalshi_all_in_sell_proceeds(bid_price: float, contracts: int, maker: bool = False) -> float:
    """Proceeds per contract to sell at bid, net of fees"""
    return bid_price - kalshi_fee_per_contract(contracts, bid_price, maker=maker)

# ============================================================================
# POLYMARKET PRICE FETCHER (lightweight, no full client needed)
# ============================================================================
class PolyPriceFetcher:
    """Lightweight fetcher for Polymarket prices without full client setup"""
    
    def __init__(self):
        self.gamma_url = "https://gamma-api.polymarket.com"
        self.clob_url = "https://clob.polymarket.com"
    
    def search_markets(self, query: str, limit: int = 10) -> list:
        """Search for markets matching a query"""
        try:
            params = {"_q": query, "_limit": str(limit)}
            res = httpx.get(f"{self.gamma_url}/markets", params=params, timeout=10.0)
            if res.status_code == 200:
                return res.json()
            return []
        except Exception as e:
            print(f"Error searching markets: {e}")
            return []
    
    def get_market_by_id(self, condition_id: str) -> Optional[dict]:
        """Get market data by condition ID"""
        try:
            res = httpx.get(f"{self.gamma_url}/markets/{condition_id}", timeout=10.0)
            if res.status_code == 200:
                return res.json()
            return None
        except Exception as e:
            print(f"Error getting market: {e}")
            return None
    
    def get_orderbook(self, token_id: str) -> Optional[dict]:
        """Get orderbook for a specific token"""
        try:
            res = httpx.get(f"{self.clob_url}/book?token_id={token_id}", timeout=10.0)
            if res.status_code == 200:
                return res.json()
            return None
        except Exception as e:
            print(f"Error getting orderbook: {e}")
            return None
    
    def get_midpoint(self, token_id: str) -> Optional[float]:
        """Get midpoint price for a token"""
        try:
            res = httpx.get(f"{self.clob_url}/midpoint?token_id={token_id}", timeout=10.0)
            if res.status_code == 200:
                data = res.json()
                return float(data.get("mid", 0))
            return None
        except Exception as e:
            print(f"Error getting midpoint: {e}")
            return None
    
    def get_price(self, token_id: str, side: str = "BUY") -> Optional[float]:
        """Get current price for a token"""
        try:
            res = httpx.get(f"{self.clob_url}/price?token_id={token_id}&side={side}", timeout=10.0)
            if res.status_code == 200:
                data = res.json()
                return float(data.get("price", 0))
            return None
        except Exception as e:
            print(f"Error getting price: {e}")
            return None

poly_fetcher = PolyPriceFetcher()
print("Polymarket price fetcher initialized")

Polymarket price fetcher initialized


In [5]:
# ============================================================================
# MARKET PAIRING: Match Kalshi markets with Polymarket equivalents
# ============================================================================
@dataclass
class MarketPair:
    """Represents a paired market between Kalshi and Polymarket"""
    kalshi_ticker: str
    kalshi_team: str  # The team this YES side represents
    poly_token_id: str  # Polymarket token ID for YES on same outcome
    poly_market_id: str
    description: str

def _to_float(x) -> Optional[float]:
    if x is None:
        return None
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

class DirectionalBot:
    """
    Polls Polymarket for price signals, executes on Kalshi when profitable.
    
    Strategy:
    - Poly price is "true" price (higher volume)
    - If Kalshi ask < Poly price - threshold: BUY on Kalshi (underpriced)
    - If Kalshi bid > Poly price + threshold: SELL on Kalshi (overpriced)
    """
    
    def __init__(
        self,
        kalshi_client,
        market_pairs: list,  # List of MarketPair objects
        poll_period_ms: int = 500,
        contract_size: int = 100,  # For fee calculation
        trade_size: int = 5,
        max_position: int = 20,
        min_edge: float = 0.02,  # Minimum edge after fees to trade (2 cents)
        maker: bool = False,  # Use taker fees by default
        auto_trade: bool = False,
        logger: logging.Logger = None,
    ):
        self.kalshi = kalshi_client
        self.poly = PolyPriceFetcher()
        self.market_pairs = market_pairs
        
        self.poll_period_s = poll_period_ms / 1000.0
        self.C = contract_size
        self.trade_size = trade_size
        self.max_position = max_position
        self.min_edge = min_edge
        self.maker = maker
        self.auto_trade = auto_trade
        
        self.logger = logger or setup_logger(NOTEBOOK_DIR)
        self.logger.info(f"DirectionalBot initialized with {len(market_pairs)} market pairs")
        self.logger.info(f"Settings: auto_trade={auto_trade}, trade_size={trade_size}, min_edge={min_edge}")
        
        # Position tracking per market
        self.positions = {}
        for pair in market_pairs:
            self.positions[f"{pair.kalshi_ticker}_yes"] = 0
            self.positions[f"{pair.kalshi_ticker}_no"] = 0
        
        self.trade_history = []
        self.last_trade_time = 0
        self.trade_cooldown = 2.0  # seconds between trades
        
        # Data tracking
        self.history = 600
        self.t0 = time.time()
        self.data = {pair.kalshi_ticker: {
            "x": deque(maxlen=self.history),
            "poly_price": deque(maxlen=self.history),
            "kalshi_yes_ask": deque(maxlen=self.history),
            "kalshi_yes_bid": deque(maxlen=self.history),
            "buy_edge": deque(maxlen=self.history),
            "sell_edge": deque(maxlen=self.history),
        } for pair in market_pairs}
        
        self.lock = threading.Lock()
        self.stop_evt = threading.Event()
        self.th = threading.Thread(target=self._run, daemon=True)
    
    def start(self):
        self.th.start()
        self.logger.info("Bot started")
        return self
    
    def stop(self):
        self.stop_evt.set()
        self.th.join(timeout=2.0)
        self.logger.info("Bot stopped")
    
    def _can_trade(self, ticker: str, side: str, quantity: int) -> bool:
        pos_key = f"{ticker}_{side}"
        new_position = self.positions.get(pos_key, 0) + quantity
        return abs(new_position) <= self.max_position
    
    def _execute_trade(self, ticker: str, side: str, action: str, price: float, quantity: int) -> bool:
        """Execute a trade on Kalshi"""
        try:
            order_params = {
                "ticker": ticker,
                "action": action,  # "buy" or "sell"
                "side": side,  # "yes" or "no"
                "count": quantity,
                "type": "market",
            }
            
            ts = datetime.now().strftime("%H:%M:%S")
            msg = f"[{ts}] EXECUTING: {action.upper()} {quantity} {side.upper()} {ticker} @ {price:.3f}"
            print(msg)
            self.logger.info(msg)
            
            response = self.kalshi.create_order(**order_params)
            
            pos_key = f"{ticker}_{side}"
            if action == "buy":
                self.positions[pos_key] = self.positions.get(pos_key, 0) + quantity
            else:
                self.positions[pos_key] = self.positions.get(pos_key, 0) - quantity
            
            trade_record = {
                "time": time.time(),
                "ticker": ticker,
                "side": side,
                "action": action,
                "quantity": quantity,
                "price": price,
                "response": str(response),
            }
            self.trade_history.append(trade_record)
            
            msg = f"[{ts}] ✓ Trade executed. Position {pos_key}: {self.positions[pos_key]}"
            print(msg)
            self.logger.info(msg)
            return True
            
        except Exception as e:
            msg = f"[TRADE ERROR] {ticker} {side} {action}: {e}"
            print(msg)
            self.logger.error(msg, exc_info=True)
            return False
    
    def _run(self):
        while not self.stop_evt.is_set():
            for pair in self.market_pairs:
                try:
                    self._process_pair(pair)
                except Exception as e:
                    self.logger.error(f"Error processing {pair.kalshi_ticker}: {e}", exc_info=True)
            
            time.sleep(self.poll_period_s)
    
    def _process_pair(self, pair: MarketPair):
        """Process a single market pair"""
        ts = datetime.now().strftime("%H:%M:%S")
        
        # Get Polymarket price (this is our "true" price signal)
        poly_price = self.poly.get_midpoint(pair.poly_token_id)
        if poly_price is None or poly_price == 0:
            # Try getting price from orderbook
            poly_price = self.poly.get_price(pair.poly_token_id, "BUY")
        
        if poly_price is None or poly_price == 0:
            self.logger.debug(f"No Poly price for {pair.kalshi_ticker}")
            return
        
        # Get Kalshi prices
        kalshi_market = self.kalshi.get_market(pair.kalshi_ticker)
        kalshi_data = kalshi_market.market.model_dump()
        
        kalshi_yes_ask = _to_float(kalshi_data.get("yes_ask_dollars"))
        kalshi_yes_bid = _to_float(kalshi_data.get("yes_bid_dollars"))
        
        if kalshi_yes_ask is None:
            self.logger.debug(f"No Kalshi ask for {pair.kalshi_ticker}")
            return
        
        # Calculate all-in costs (with fees)
        buy_cost = kalshi_all_in_buy_cost(kalshi_yes_ask, self.C, self.maker)
        sell_proceeds = kalshi_all_in_sell_proceeds(kalshi_yes_bid, self.C, self.maker) if kalshi_yes_bid else 0
        
        # Calculate edges:
        # BUY edge: If Poly says it's worth X, and Kalshi all-in cost is Y, edge = X - Y
        # Positive buy_edge means Kalshi is cheap relative to Poly
        buy_edge = poly_price - buy_cost
        
        # SELL edge: If Poly says it's worth X, and Kalshi sell proceeds is Y, edge = Y - X
        # Positive sell_edge means Kalshi is rich relative to Poly
        sell_edge = sell_proceeds - poly_price if sell_proceeds > 0 else float("-inf")
        
        # Store data
        now = time.time()
        with self.lock:
            self.data[pair.kalshi_ticker]["x"].append(now - self.t0)
            self.data[pair.kalshi_ticker]["poly_price"].append(poly_price)
            self.data[pair.kalshi_ticker]["kalshi_yes_ask"].append(kalshi_yes_ask)
            self.data[pair.kalshi_ticker]["kalshi_yes_bid"].append(kalshi_yes_bid)
            self.data[pair.kalshi_ticker]["buy_edge"].append(buy_edge)
            self.data[pair.kalshi_ticker]["sell_edge"].append(sell_edge)
        
        # Log state periodically
        if len(self.data[pair.kalshi_ticker]["x"]) % 20 == 0:
            self.logger.debug(
                f"{pair.kalshi_ticker}: Poly={poly_price:.3f}, "
                f"Kalshi ask/bid={kalshi_yes_ask:.3f}/{kalshi_yes_bid if kalshi_yes_bid else 'None'}, "
                f"buy_edge={buy_edge:.4f}, sell_edge={sell_edge:.4f}"
            )
        
        # Trading signals
        if buy_edge >= self.min_edge:
            msg = f"[{ts}] 📈 BUY SIGNAL {pair.kalshi_ticker}: Poly={poly_price:.3f}, Kalshi ask={kalshi_yes_ask:.3f}, edge={buy_edge:.4f}"
            print(msg)
            self.logger.info(msg)
            
            if self.auto_trade:
                if time.time() - self.last_trade_time >= self.trade_cooldown:
                    if self._can_trade(pair.kalshi_ticker, "yes", self.trade_size):
                        self.logger.info(f"Executing BUY on {pair.kalshi_ticker}")
                        if self._execute_trade(pair.kalshi_ticker, "yes", "buy", kalshi_yes_ask, self.trade_size):
                            self.last_trade_time = time.time()
                    else:
                        self.logger.warning(f"Position limit reached for {pair.kalshi_ticker}")
                else:
                    self.logger.debug("Trade cooldown active")
        
        if sell_edge >= self.min_edge and kalshi_yes_bid is not None:
            msg = f"[{ts}] 📉 SELL SIGNAL {pair.kalshi_ticker}: Poly={poly_price:.3f}, Kalshi bid={kalshi_yes_bid:.3f}, edge={sell_edge:.4f}"
            print(msg)
            self.logger.info(msg)
            
            if self.auto_trade:
                current_pos = self.positions.get(f"{pair.kalshi_ticker}_yes", 0)
                if current_pos > 0 and time.time() - self.last_trade_time >= self.trade_cooldown:
                    sell_qty = min(self.trade_size, current_pos)
                    self.logger.info(f"Executing SELL on {pair.kalshi_ticker}")
                    if self._execute_trade(pair.kalshi_ticker, "yes", "sell", kalshi_yes_bid, sell_qty):
                        self.last_trade_time = time.time()

print("DirectionalBot class defined")

DirectionalBot class defined


## ⚠️ Important: Market Overlap

**Polymarket does NOT have daily sports betting** (NHL, NBA, NFL games). They focus on:
- 🗳️ Politics (elections, policy)
- 💰 Crypto (Bitcoin price, ETF approvals)
- 🌍 Current Events (geopolitics, Fed decisions)

**For this directional strategy**, you need to find markets that exist on **BOTH** Kalshi and Polymarket. 

### Options:
1. **Find overlapping markets** - Search for politics/crypto/events on both platforms
2. **NHL-only strategy** - Use a different approach for NHL (no cross-platform signal available)

In [6]:
# Get Kalshi NHL markets
nhl_markets = kalshi_wrapped.GetAllNHLMarkets(status="open")
print(f"Found {len(nhl_markets)} NHL markets on Kalshi")

# Show first few markets
for m in nhl_markets[:6]:
    d = m.model_dump()
    print(f"  {d['ticker']}: {d['yes_sub_title']} - YES ask: {d.get('yes_ask_dollars')}")

Found 44 NHL markets on Kalshi
  KXNHLGAME-26JAN22DETMIN-MIN: MIN Wild - YES ask: 0.5700
  KXNHLGAME-26JAN22DETMIN-DET: DET Red Wings - YES ask: 0.4600
  KXNHLGAME-26JAN22FLAWPG-WPG: WPG Jets - YES ask: 0.4900
  KXNHLGAME-26JAN22FLAWPG-FLA: FLA Panthers - YES ask: 0.5500
  KXNHLGAME-26JAN22PITEDM-PIT: PIT Penguins - YES ask: 0.4000
  KXNHLGAME-26JAN22PITEDM-EDM: EDM Oilers - YES ask: 0.6400


In [ ]:
# The Gamma API search seems broken (returning stale 2020 data)
# Let's try fetching current/active markets directly

import ast

def parse_prices(price_str):
    """Safely parse outcome prices from Polymarket"""
    try:
        if not price_str:
            return [0, 0]
        prices = ast.literal_eval(price_str)
        return [float(p) for p in prices]
    except:
        return [0, 0]

# Try different Gamma API endpoints for active markets
print("Trying to fetch active Polymarket markets...")

# Method 1: Get markets with active=true filter
try:
    params = {"active": "true", "closed": "false", "_limit": "50"}
    res = httpx.get("https://gamma-api.polymarket.com/markets", params=params, timeout=15.0)
    if res.status_code == 200:
        markets = res.json()
        # Filter for markets with real trading activity
        current = [m for m in markets if parse_prices(m.get("outcomePrices"))[0] > 0.02 
                   and parse_prices(m.get("outcomePrices"))[0] < 0.98]
        print(f"\n✓ Found {len(current)} active markets with prices between 2-98%:")
        for m in current[:8]:
            prices = parse_prices(m.get("outcomePrices"))
            print(f"  [{m.get('id')}] {m.get('question', '')[:60]}...")
            print(f"       YES: {prices[0]:.3f}, Volume: {m.get('volume', 'N/A')}")
    else:
        print(f"API returned status {res.status_code}")
except Exception as e:
    print(f"Error: {e}")

# Method 2: Try the events endpoint
print("\n--- Trying events endpoint ---")
try:
    res = httpx.get("https://gamma-api.polymarket.com/events?active=true&_limit=20", timeout=15.0)
    if res.status_code == 200:
        events = res.json()
        print(f"Found {len(events)} active events:")
        for e in events[:5]:
            print(f"  {e.get('title', '')[:60]}... (markets: {len(e.get('markets', []))})")
except Exception as e:
    print(f"Events error: {e}")


'Trump' → 3 active markets:
  What will the total value locked (TVL) in DeFi be at the end of 2020, ...
     YES: 0.584 | ID: 43
     Token: 393000008017503135064158819624...
  What will the USD price of Filecoin ($FIL) be on November 17th, 2020?...
     YES: 0.499 | ID: 44
     Token: 107699364100754617875486132785...

'Bitcoin' → 3 active markets:
  What will the total value locked (TVL) in DeFi be at the end of 2020, ...
     YES: 0.584 | ID: 43
     Token: 393000008017503135064158819624...
  What will the USD price of Filecoin ($FIL) be on November 17th, 2020?...
     YES: 0.499 | ID: 44
     Token: 107699364100754617875486132785...

'Fed' → 3 active markets:
  What will the total value locked (TVL) in DeFi be at the end of 2020, ...
     YES: 0.584 | ID: 43
     Token: 393000008017503135064158819624...
  What will the USD price of Filecoin ($FIL) be on November 17th, 2020?...
     YES: 0.499 | ID: 44
     Token: 107699364100754617875486132785...

'inflation' → 3 active markets:
 

## Step 2: Configure Market Pairs

You need to manually configure the market pairs between Kalshi and Polymarket. 
The key is matching:
- Kalshi ticker (e.g., "KXNHLGAME-25JAN20-TBL")
- Polymarket token ID for the YES outcome of the SAME team/event

In [8]:
# ============================================================================
# CONFIGURE YOUR MARKET PAIRS HERE
# ============================================================================
# You need to find markets that exist on BOTH platforms for the SAME event
# and extract the Polymarket token ID for the YES outcome

# Example configuration (you'll need to update these with real token IDs):
market_pairs = [
    # MarketPair(
    #     kalshi_ticker="KXNHLGAME-25JAN20-TBL",  # Kalshi ticker
    #     kalshi_team="Lightning",                 # Team this YES represents
    #     poly_token_id="YOUR_POLY_TOKEN_ID",      # Polymarket token ID for YES on Lightning
    #     poly_market_id="YOUR_POLY_MARKET_ID",    # Polymarket market/condition ID
    #     description="Lightning vs Opponent",
    # ),
]

# Helper: Get token ID from market search result
def extract_poly_token_id(market_data: dict, yes_outcome: bool = True) -> str:
    """Extract YES or NO token ID from Polymarket market data"""
    import ast
    token_ids = market_data.get("clobTokenIds", "[]")
    if isinstance(token_ids, str):
        token_ids = ast.literal_eval(token_ids)
    # Index 0 is typically YES, index 1 is NO (but verify for your specific market)
    return token_ids[0] if yes_outcome else token_ids[1]

print("Market pairs configuration ready")
print(f"Currently configured: {len(market_pairs)} pairs")
print("\n⚠️  You need to manually configure market_pairs with matching Kalshi-Polymarket markets")

Market pairs configuration ready
Currently configured: 0 pairs

⚠️  You need to manually configure market_pairs with matching Kalshi-Polymarket markets


## Step 3: Run the Bot

Once you have configured market pairs, run the bot to:
1. Poll Polymarket for prices (every 500ms by default)
2. Compare to Kalshi prices
3. Execute trades when edge > min_edge (after fees)

In [ ]:
# Run the directional bot
# Set auto_trade=True to enable automatic trading (BE CAREFUL!)

if len(market_pairs) > 0:
    bot = DirectionalBot(
        kalshi_client=kalshi_wrapped.GetClient(),
        market_pairs=market_pairs,
        poll_period_ms=500,
        contract_size=100,       # For fee calculation
        trade_size=5,            # Contracts per trade
        max_position=20,         # Max position per side
        min_edge=0.02,           # Minimum edge to trade (2 cents after fees)
        maker=False,             # Use taker fees (7%)
        auto_trade=False,        # ← SET TO True TO ENABLE LIVE TRADING
    ).start()
    
    print("Bot running! Check logs for signals.")
    print("To stop: bot.stop()")
else:
    print("❌ No market pairs configured. See Step 2 to configure matching markets.")
    bot = None

In [ ]:
# Check positions and trade history
if bot:
    print("Current Positions:")
    for pos, qty in bot.positions.items():
        if qty != 0:
            print(f"  {pos}: {qty} contracts")
    
    if not any(bot.positions.values()):
        print("  (No positions)")
    
    print(f"\nTrade History: {len(bot.trade_history)} trades")
    for trade in bot.trade_history[-5:]:
        print(f"  {trade['action'].upper()} {trade['quantity']} {trade['side']} {trade['ticker']} @ {trade['price']:.3f}")